# DTFA on Google Colab (VOC-Fog)

Runtime → Change runtime type → **GPU** (T4 or better).

Before running: upload `DTFA_colab.zip` (in your DTFA folder) to the top of **My Drive**.
All checkpoints and logs go to `My Drive/DTFA_logs`, so they survive a Colab disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Unpack the repo (code + datasets + yolox_s.pth) onto Colab's local disk

In [ ]:
!unzip -q -o /content/drive/MyDrive/DTFA_colab.zip -d /content/
%cd /content/DTFA
!mkdir -p /content/drive/MyDrive/DTFA_logs && rm -rf logs && ln -s /content/drive/MyDrive/DTFA_logs logs
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Colab's preinstalled PyTorch works; do NOT install requirements.txt (its pins don't build on Colab's Python).
!pip -q install tensorboard
import os
os.environ['PYTHONPATH'] = '/content/DTFA'

### (Alternative to the zip) rebuild the data inside Colab
If you upload only the code, this downloads VOC2012 from GitHub and regenerates the exact same clean/fog images (~3 min).

In [ ]:
# !curl -sSL -o /content/voc.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VOCtrainval_11-May-2012.zip
# !cd /content && unzip -q voc.zip 'VOCdevkit/VOC2012/JPEGImages/*' 'VOCdevkit/VOC2012/Annotations/*'
# !python tools/prepare_data.py --voc /content/VOCdevkit/VOC2012
# !mkdir -p model_data && curl -sSL -o model_data/yolox_s.pth https://github.com/bubbliiiing/yolox-pytorch/releases/download/v1.0/yolox_s.pth

## 2. Teachers (SPT / IRT)
The zip already contains the authors' `model_data/SPT.pth` and `model_data/IRT.pth`, so the cell below
**skips training** and just confirms they are there. It only trains a teacher if its file is missing
(resume a teacher run with `--init_epoch N --model_path logs/SPT/epNNN-....pth`).

In [ ]:
import os
for t in ['SPT', 'IRT']:
    if os.path.exists(f'model_data/{t}.pth'):
        print(f'model_data/{t}.pth found -> skipping {t} pre-training')
    else:
        print(f'model_data/{t}.pth missing -> training {t} teacher')
        !python train/pretrain_{t}.py --num_workers 2 --save_period 5 --logs_dir logs/{t}/
        !cp "$(ls -t logs/{t}/*.pth | head -1)" model_data/{t}.pth
!ls -la model_data

## 3. Train the adverse-weather student (AWD)
Should print `matched 215/215 teacher tensors` twice. Checkpoints: `logs/epNNN-loss...pth`.

To resume: `!INIT_EPOCH=N MODEL_PATH=logs/epNNN-....pth SAVE_PERIOD=5 NUM_WORKERS=2 python train/train_AWD.py`

In [ ]:
!SAVE_PERIOD=5 NUM_WORKERS=2 python train/train_AWD.py

## 4. Test: mAP@0.5 on VOC-Fog test and VOC-Clean test

In [ ]:
!cp "$(ls -t logs/ep*.pth | head -1)" model_data/AWD.pth
!python get_map.py
!TEST_DIR=datasets/voc_clean/test MAP_OUT=logs/clean_map_out python get_map.py